# CNN Image Classification: Reproducible Comparison

Compare a shared baseline CNN on MNIST and CIFAR-10, then evaluate the locally collected Chairs vs Shoes experiment. Core baseline architecture is imported from `src.models`.

In [ ]:
from pathlib import Path
import os, random, sys
SEED=42
os.environ["PYTHONHASHSEED"]=str(SEED)
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/"src").exists():
    PROJECT_ROOT=PROJECT_ROOT.parent
sys.path.insert(0,str(PROJECT_ROOT))
from src.models import build_baseline_cnn
import pandas as pd
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

## MNIST

The baseline CNN is trained for 10 epochs with Adam, sparse categorical cross-entropy, batch size 128, and seed 42.

In [ ]:
from tensorflow.keras.datasets import mnist
(x_train,y_train),(x_test,y_test)=mnist.load_data()
x_train=x_train[...,None].astype("float32")/255.0
x_test=x_test[...,None].astype("float32")/255.0

mnist_model=build_baseline_cnn(x_train.shape[1:],10)
mnist_model.fit(x_train,y_train,epochs=10,batch_size=128,verbose=0)
mnist_loss,mnist_acc=mnist_model.evaluate(x_test,y_test,verbose=0)
print(f"MNIST test accuracy: {mnist_acc:.4%}")

## CIFAR-10

In [ ]:
from tensorflow.keras.datasets import cifar10
(x_train,y_train),(x_test,y_test)=cifar10.load_data()
y_train,y_test=y_train.ravel(),y_test.ravel()
x_train,x_test=x_train.astype("float32")/255.0,x_test.astype("float32")/255.0

cifar_model=build_baseline_cnn(x_train.shape[1:],10)
cifar_model.fit(x_train,y_train,epochs=10,batch_size=128,verbose=0)
cifar_loss,cifar_acc=cifar_model.evaluate(x_test,y_test,verbose=0)
print(f"CIFAR-10 test accuracy: {cifar_acc:.4%}")

## Local Chairs vs Shoes

The supplied local dataset contains 36 training images and 8 test images. Because the test set is very small, its accuracy should be treated as descriptive rather than as a stable estimate of generalization.

In [ ]:
# Expected structure:
# data/local_chairs_shoes/Train/{Chairs,Shoes}
# data/local_chairs_shoes/Test/{Chairs,Shoes}
#
# The original assignment used a deeper augmented CNN for this experiment.
# Re-run that section here after placing the local dataset in the expected path.

In [ ]:
summary=pd.DataFrame([
    ["MNIST",0.9899,"9899/10000","Baseline CNN"],
    ["CIFAR-10",0.6734,"6734/10000","Baseline CNN"],
    ["Chairs vs Shoes",0.8750,"7/8","Augmented 3-block CNN"],
],columns=["Dataset","Test Accuracy","Correct / Total","Architecture"])
display(summary)
summary.to_csv(PROJECT_ROOT/"results/tables/cnn_dataset_comparison.csv",index=False)

fig,ax=plt.subplots(figsize=(7,4))
ax.bar(summary["Dataset"],summary["Test Accuracy"]*100)
ax.set_ylabel("Test accuracy (%)")
ax.set_ylim(0,100)
ax.set_title("CNN Test Accuracy")
ax.tick_params(axis="x",rotation=20)
fig.tight_layout()
fig.savefig(PROJECT_ROOT/"results/figures/cnn_test_accuracy.svg")
plt.show()